# Complete Guide to Bayesian Uncertainty Quantification

This notebook provides a comprehensive introduction to Bayesian deep learning using `incerto`. We'll cover:

1. **Understanding Bayesian Uncertainty** - Epistemic vs Aleatoric uncertainty
2. **Setup and Data Preparation** - Load Fashion-MNIST with optimized loaders
3. **MC Dropout** - Approximate Bayesian inference via dropout
4. **Deep Ensembles** - Multiple models with different initializations
5. **SWAG** - Stochastic Weight Averaging-Gaussian
6. **Laplace Approximation** - Post-hoc Bayesian inference
7. **Variational Bayesian NN** - Weight distributions via variational inference
8. **Compare Methods** - Evaluate uncertainty quality
9. **Uncertainty Decomposition** - Epistemic vs Aleatoric
10. **Production Deployment** - Save and deploy Bayesian models

**Runtime:** ~5 min (MPS/CUDA), ~15 min (CPU)

## Part 1: Understanding Bayesian Uncertainty

### Types of Uncertainty

**Epistemic Uncertainty (Model Uncertainty)**:
- Uncertainty about the model parameters
- Reducible with more training data
- High for OOD samples or regions with sparse data
- Example: Model hasn't seen images like this before

**Aleatoric Uncertainty (Data Uncertainty)**:
- Inherent noise in the data
- Irreducible even with infinite data
- High for noisy or ambiguous inputs
- Example: Blurry image that could be multiple classes

### Methods in This Notebook

| Method | Type | Key Idea | Training Cost | Inference Cost |
|--------|------|----------|---------------|----------------|
| **MC Dropout** | Approximate | Dropout at inference time | 1x | 10-20x |
| **Deep Ensembles** | Ensemble | Train multiple models | 5-10x | 5-10x |
| **SWAG** | Approximate | Bayesian model averaging | ~1.2x | 10-30x |
| **Laplace** | Post-hoc | Gaussian around MAP | 1x + Hessian | 10x |
| **Variational** | Learned | Weight distributions | 1-2x | 10x |

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Bayesian methods
from incerto.bayesian import (
    MCDropout,
    DeepEnsemble,
    SWAG,
    LaplaceApproximation,
    VariationalBayesNN,
)

# Uncertainty utilities
from incerto.bayesian import (
    decompose_uncertainty,
    uncertainty_quality,
)

from incerto.utils import seed_everything

seed_everything(42)

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## Part 2: Setup and Data Preparation

We use **Fashion-MNIST** - a harder drop-in replacement for MNIST with the same
format (28x28 grayscale, 10 classes). Unlike MNIST where CNNs reach 99%+ accuracy,
Fashion-MNIST (~90% accuracy) has more ambiguous samples, making it ideal for
demonstrating uncertainty estimation.

In [ ]:
# Load Fashion-MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

# Split: 50k train, 10k validation (FashionMNIST has 60k train samples)
train_size = 50000
val_size = 10000
train_subset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Optimized data loaders
num_workers = min(4, os.cpu_count() or 0)
pin_memory = device.type == "cuda"
loader_kwargs = dict(num_workers=num_workers, pin_memory=pin_memory, persistent_workers=num_workers > 0)

train_loader = DataLoader(train_subset, batch_size=256, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, **loader_kwargs)

# Class names for visualization
CLASS_NAMES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

print(f"Training: {len(train_subset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"DataLoader: num_workers={num_workers}, pin_memory={pin_memory}")

In [ ]:
# Define CNN with dropout (required for MC Dropout)
class DropoutCNN(nn.Module):
    """CNN with dropout layers for MC Dropout uncertainty estimation."""
    def __init__(self, num_classes=10, dropout_rate=0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(dropout_rate),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(dropout_rate),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

# Training helper
def train_model(model, train_loader, epochs=5, lr=0.001, device="cpu", verbose=True):
    """Train a model and return final accuracy."""
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        if verbose and (epoch + 1) % 2 == 0:
            acc = 100. * correct / total
            print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Acc = {acc:.1f}%")
    
    return 100. * correct / total

print("Model and training utilities defined.")

## Part 3: MC Dropout - Approximate Bayesian Inference

MC Dropout treats dropout as Bayesian inference. By keeping dropout active at test time
and running multiple forward passes, we approximate the posterior distribution over models.

**Key insight**: Dropout is equivalent to a deep Gaussian process (Gal & Ghahramani, 2016)

In [ ]:
# Train base model with dropout
print("Training MC Dropout base model...")
seed_everything(42)
mc_base_model = DropoutCNN(dropout_rate=0.25).to(device)
train_model(mc_base_model, train_loader, epochs=5, device=device)
print("Done!")

In [ ]:
# Create MC Dropout wrapper
mc_dropout = MCDropout(
    model=mc_base_model,
    num_samples=20,    # Number of Monte Carlo samples
    dropout_rate=0.25  # Can override model's dropout rate
)

print("MC Dropout configured:")
print(f"  - Samples per prediction: {mc_dropout.num_samples}")
print(f"  - Dropout rate: {mc_dropout.dropout_rate}")

In [ ]:
# Get predictions with uncertainty on test set
all_mc_means = []
all_mc_vars = []
all_labels = []

for inputs, labels in test_loader:
    inputs = inputs.to(device)
    
    # MCDropout.predict returns (mean, variance) tuple
    mean, variance = mc_dropout.predict(inputs)
    
    all_mc_means.append(mean.cpu())
    all_mc_vars.append(variance.cpu())
    all_labels.append(labels)

mc_means = torch.cat(all_mc_means)
mc_vars = torch.cat(all_mc_vars)
test_labels = torch.cat(all_labels)

# Compute accuracy
mc_preds = mc_means.argmax(dim=1)
mc_acc = (mc_preds == test_labels).float().mean().item() * 100

# Uncertainty = mean variance across classes
mc_uncertainty = mc_vars.mean(dim=1)

print("MC Dropout Results:")
print(f"  - Accuracy: {mc_acc:.2f}%")
print(f"  - Mean uncertainty: {mc_uncertainty.mean():.4f}")
print(f"  - Predictions shape: {mc_means.shape}")

## Part 4: Deep Ensembles - Gold Standard

Deep Ensembles train multiple models with different random initializations.
The diversity between models provides high-quality uncertainty estimates.

**Reference**: Lakshminarayanan et al., "Simple and Scalable Predictive Uncertainty" (NeurIPS 2017)

In [ ]:
# Create ensemble with a model factory function
def create_model():
    return DropoutCNN(dropout_rate=0.0)  # No dropout for ensemble

ensemble = DeepEnsemble(
    model_fn=create_model,
    num_models=5
)

print(f"Deep Ensemble: {len(ensemble.models)} models")

In [ ]:
# Train each ensemble member with different random seed
print("Training Deep Ensemble (5 models)...")

for i, model in enumerate(ensemble.models):
    seed_everything(42 + i)  # Different seed for each model
    print(f"  Model {i+1}/5:")
    train_model(model, train_loader, epochs=5, device=device, verbose=True)

print("Done!")

In [ ]:
# Get ensemble predictions
all_ens_means = []
all_ens_vars = []

for inputs, labels in test_loader:
    inputs = inputs.to(device)
    
    # DeepEnsemble.predict returns (mean, variance) or (mean, variance, samples)
    mean, variance = ensemble.predict(inputs)
    
    all_ens_means.append(mean.cpu())
    all_ens_vars.append(variance.cpu())

ens_means = torch.cat(all_ens_means)
ens_vars = torch.cat(all_ens_vars)

# Compute accuracy
ens_preds = ens_means.argmax(dim=1)
ens_acc = (ens_preds == test_labels).float().mean().item() * 100

# Uncertainty = mean variance across classes
ens_uncertainty = ens_vars.mean(dim=1)

print("Deep Ensemble Results:")
print(f"  - Accuracy: {ens_acc:.2f}%")
print(f"  - Mean uncertainty: {ens_uncertainty.mean():.4f}")

## Part 5: SWAG - Stochastic Weight Averaging Gaussian

SWAG approximates the Bayesian posterior by fitting a Gaussian to the SGD trajectory.
It collects weight statistics during training and samples from this distribution at test time.

**Reference**: Maddox et al., "A Simple Baseline for Bayesian Uncertainty" (NeurIPS 2019)

In [ ]:
# Train model and collect SWAG statistics
print("Training SWAG model...")

seed_everything(42)
swag_base_model = DropoutCNN(dropout_rate=0.0).to(device)

# Create SWAG wrapper
swag = SWAG(
    model=swag_base_model,
    num_samples=20,   # Samples for prediction
    var_clamp=1e-6    # Variance clamping for stability
)

optimizer = torch.optim.SGD(swag_base_model.parameters(), lr=0.01, momentum=0.9)

# Train for a few epochs, then start collecting SWAG statistics
warmup_epochs = 3
swag_epochs = 5
total_epochs = warmup_epochs + swag_epochs

for epoch in range(total_epochs):
    swag_base_model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = swag_base_model(inputs)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    acc = 100. * correct / total
    
    # Collect SWAG statistics after warmup
    if epoch >= warmup_epochs:
        swag.collect_model(swag_base_model)
        print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Acc = {acc:.1f}% [SWAG collected]")
    else:
        print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Acc = {acc:.1f}%")

print(f"Done! Collected {swag.n_models} model snapshots.")

In [ ]:
# Get SWAG predictions
all_swag_means = []
all_swag_vars = []

for inputs, labels in test_loader:
    inputs = inputs.to(device)
    
    # SWAG.predict samples from the approximate posterior
    mean, variance = swag.predict(inputs)
    
    all_swag_means.append(mean.cpu())
    all_swag_vars.append(variance.cpu())

swag_means = torch.cat(all_swag_means)
swag_vars = torch.cat(all_swag_vars)

# Compute accuracy
swag_preds = swag_means.argmax(dim=1)
swag_acc = (swag_preds == test_labels).float().mean().item() * 100

# Uncertainty
swag_uncertainty = swag_vars.mean(dim=1)

print("SWAG Results:")
print(f"  - Accuracy: {swag_acc:.2f}%")
print(f"  - Mean uncertainty: {swag_uncertainty.mean():.4f}")

## Part 6: Laplace Approximation

Laplace Approximation fits a Gaussian posterior around the MAP estimate (trained weights)
using the Hessian of the loss. It's a post-hoc method that works with pre-trained models.

**Key hyperparameter**: `prior_precision` controls posterior variance. Higher values mean
smaller weight perturbations and predictions closer to the MAP estimate. Too low and the
network is destroyed by noise; too high and you get no uncertainty. Typical values: 100-10000.

**Reference**: MacKay (1992); Daxberger et al., "Laplace Redux" (NeurIPS 2021)

In [ ]:
# Train a model normally first
print("Training Laplace base model...")
seed_everything(42)
laplace_base_model = DropoutCNN(dropout_rate=0.0).to(device)
train_model(laplace_base_model, train_loader, epochs=5, device=device)
print("Done!")

In [ ]:
# Create Laplace approximation and fit on validation data
# Note: prior_precision controls how much we trust the prior vs. the data
# Higher values = smaller weight perturbations = predictions closer to MAP
laplace = LaplaceApproximation(
    model=laplace_base_model,
    likelihood='classification',
    prior_precision=1000.0,  # High precision = small posterior variance
    num_samples=20
)

# Fit Laplace approximation (computes Hessian diagonal)
print("Fitting Laplace approximation (computing Hessian)...")
laplace.fit(val_loader, device=str(device))
print("Done!")

In [ ]:
# Get Laplace predictions
all_lap_means = []
all_lap_vars = []

for inputs, labels in test_loader:
    inputs = inputs.to(device)
    
    # LaplaceApproximation.predict samples from the posterior
    mean, variance = laplace.predict(inputs)
    
    all_lap_means.append(mean.cpu())
    all_lap_vars.append(variance.cpu())

lap_means = torch.cat(all_lap_means)
lap_vars = torch.cat(all_lap_vars)

# Compute accuracy
lap_preds = lap_means.argmax(dim=1)
lap_acc = (lap_preds == test_labels).float().mean().item() * 100

# Uncertainty
lap_uncertainty = lap_vars.mean(dim=1)

print("Laplace Approximation Results:")
print(f"  - Accuracy: {lap_acc:.2f}%")
print(f"  - Mean uncertainty: {lap_uncertainty.mean():.4f}")

## Part 7: Variational Bayesian Neural Network

Variational BNNs learn a distribution over weights using variational inference.
Each weight has a learned mean and variance (Bayes by Backprop).

**Key hyperparameter**: `prior_std` controls the prior distribution over weights. The weight
variances are initialized to match the prior, so the initial KL divergence is near zero.
The network learns to adjust these variances during training.

**Reference**: Blundell et al., "Weight Uncertainty in Neural Networks" (ICML 2015)

In [ ]:
# Create and train Variational BNN
# Note: VBNN is a fully-connected network, so we flatten the images

seed_everything(42)
vbnn = VariationalBayesNN(
    in_features=28*28,
    hidden_sizes=[256, 128],
    out_features=10,
    prior_std=0.1,  # Small prior prevents noisy activations
    num_samples=10
).to(device)

# Higher learning rate helps convergence
optimizer = torch.optim.Adam(vbnn.parameters(), lr=0.01)

print("Training Variational BNN...")
for epoch in range(15):
    vbnn.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.view(-1, 28*28).to(device)  # Flatten
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Use variational loss (ELBO)
        loss = vbnn.variational_loss(inputs, labels, num_samples=3, kl_weight=1/len(train_loader.dataset))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        with torch.no_grad():
            outputs = vbnn(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    if (epoch + 1) % 3 == 0:
        acc = 100. * correct / total
        print(f"  Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Acc = {acc:.1f}%")

print("Done!")

In [ ]:
# Get VBNN predictions
all_vbnn_means = []
all_vbnn_vars = []

for inputs, labels in test_loader:
    inputs = inputs.view(-1, 28*28).to(device)  # Flatten
    
    # VariationalBayesNN.predict samples from weight distributions
    mean, variance = vbnn.predict(inputs)
    
    all_vbnn_means.append(mean.cpu())
    all_vbnn_vars.append(variance.cpu())

vbnn_means = torch.cat(all_vbnn_means)
vbnn_vars = torch.cat(all_vbnn_vars)

# Compute accuracy
vbnn_preds = vbnn_means.argmax(dim=1)
vbnn_acc = (vbnn_preds == test_labels).float().mean().item() * 100

# Uncertainty
vbnn_uncertainty = vbnn_vars.mean(dim=1)

print("Variational BNN Results:")
print(f"  - Accuracy: {vbnn_acc:.2f}%")
print(f"  - Mean uncertainty: {vbnn_uncertainty.mean():.4f}")

## Part 8: Compare Methods

We'll compare all five Bayesian methods on:
1. **Accuracy** - How well does the method classify?
2. **Uncertainty Quality** - Is uncertainty higher for incorrect predictions?

In [ ]:
# Collect results
results = {
    'MC Dropout': {'means': mc_means, 'uncertainty': mc_uncertainty, 'acc': mc_acc},
    'Deep Ensemble': {'means': ens_means, 'uncertainty': ens_uncertainty, 'acc': ens_acc},
    'SWAG': {'means': swag_means, 'uncertainty': swag_uncertainty, 'acc': swag_acc},
    'Laplace': {'means': lap_means, 'uncertainty': lap_uncertainty, 'acc': lap_acc},
    'Variational BNN': {'means': vbnn_means, 'uncertainty': vbnn_uncertainty, 'acc': vbnn_acc},
}

print("Method Comparison:")
print("=" * 70)
print(f"{'Method':<20} {'Accuracy':<12} {'Mean Unc.':<15} {'Unc. Quality (AUROC)'}")
print("=" * 70)

for name, data in results.items():
    preds = data['means'].argmax(dim=1)
    errors = (preds != test_labels).float()
    
    # Compute uncertainty quality (AUROC for detecting errors)
    corr, auroc = uncertainty_quality(data['uncertainty'], errors)
    
    print(f"{name:<20} {data['acc']:>6.2f}%      {data['uncertainty'].mean():>10.4f}      {auroc:.4f}")

print("=" * 70)
print("\nHigher AUROC = better uncertainty quality (uncertainty predicts errors)")

In [ ]:
# Visualize uncertainty vs correctness
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, data) in enumerate(results.items()):
    preds = data['means'].argmax(dim=1)
    correct = (preds == test_labels)
    
    unc = data['uncertainty'].numpy()
    
    axes[idx].hist(unc[correct], bins=50, alpha=0.6, label='Correct', density=True, color='green')
    axes[idx].hist(unc[~correct], bins=50, alpha=0.6, label='Incorrect', density=True, color='red')
    axes[idx].set_xlabel('Uncertainty')
    axes[idx].set_ylabel('Density')
    axes[idx].set_title(f'{name}\nAcc: {data["acc"]:.1f}%')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

# Hide the 6th subplot
axes[5].axis('off')

plt.suptitle('Uncertainty Distribution: Correct vs Incorrect Predictions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterpretation: Good uncertainty estimates show higher uncertainty for incorrect predictions.")

In [ ]:
# Visualize high-uncertainty predictions
fig, axes = plt.subplots(5, 5, figsize=(15, 15))

for method_idx, (name, data) in enumerate(results.items()):
    # Get top 5 uncertain samples
    top_uncertain_idx = torch.argsort(data['uncertainty'], descending=True)[:5]
    
    for img_idx, sample_idx in enumerate(top_uncertain_idx):
        img, label = test_dataset[sample_idx]
        pred_probs = data['means'][sample_idx]
        pred_class = pred_probs.argmax().item()
        unc = data['uncertainty'][sample_idx].item()
        
        axes[method_idx, img_idx].imshow(img.squeeze(), cmap='gray')
        
        title = f'True: {CLASS_NAMES[label][:6]}\nPred: {CLASS_NAMES[pred_class][:6]}\nUnc: {unc:.3f}'
        color = 'green' if pred_class == label else 'red'
        axes[method_idx, img_idx].set_title(title, fontsize=8, color=color)
        axes[method_idx, img_idx].axis('off')
    
    # Add method name on the left
    axes[method_idx, 0].set_ylabel(name, fontsize=10, fontweight='bold', rotation=90, labelpad=10)

plt.suptitle('Highest Uncertainty Predictions by Method', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 9: Uncertainty Decomposition

For ensemble methods, we can decompose total uncertainty into:
- **Epistemic** (model) uncertainty - reducible with more data
- **Aleatoric** (data) uncertainty - irreducible noise

Total Uncertainty = Epistemic + Aleatoric

H[y|x] = I[y;theta|x] + E[H[y|x,theta]]

In [ ]:
# Get ensemble predictions with all samples for decomposition
sample_batch = next(iter(test_loader))[0][:100].to(device)

# Get all ensemble predictions (return_samples=True gives individual model outputs)
_, _, ens_samples = ensemble.predict(sample_batch, return_samples=True)

# Decompose uncertainty using incerto utilities
total, epistemic, aleatoric = decompose_uncertainty(ens_samples.cpu())

print("Uncertainty Decomposition (first 100 test samples):")
print(f"  Total (predictive entropy): {total.mean():.4f} +/- {total.std():.4f}")
print(f"  Epistemic (mutual info):    {epistemic.mean():.4f} +/- {epistemic.std():.4f}")
print(f"  Aleatoric (expected ent):   {aleatoric.mean():.4f} +/- {aleatoric.std():.4f}")
print(f"\n  Verification: Total = Epistemic + Aleatoric: {torch.allclose(total, epistemic + aleatoric, atol=1e-5)}")

In [ ]:
# Visualize uncertainty decomposition
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(total.numpy(), bins=30, alpha=0.7, color='blue')
axes[0].set_xlabel('Uncertainty')
axes[0].set_ylabel('Count')
axes[0].set_title('Total Uncertainty\n(Predictive Entropy)')
axes[0].axvline(total.mean(), color='red', linestyle='--', label=f'Mean: {total.mean():.3f}')
axes[0].legend()

axes[1].hist(epistemic.numpy(), bins=30, alpha=0.7, color='orange')
axes[1].set_xlabel('Uncertainty')
axes[1].set_title('Epistemic Uncertainty\n(Mutual Information)')
axes[1].axvline(epistemic.mean(), color='red', linestyle='--', label=f'Mean: {epistemic.mean():.3f}')
axes[1].legend()

axes[2].hist(aleatoric.numpy(), bins=30, alpha=0.7, color='green')
axes[2].set_xlabel('Uncertainty')
axes[2].set_title('Aleatoric Uncertainty\n(Expected Entropy)')
axes[2].axvline(aleatoric.mean(), color='red', linestyle='--', label=f'Mean: {aleatoric.mean():.3f}')
axes[2].legend()

plt.tight_layout()
plt.show()

## Part 10: Production Deployment

All Bayesian methods use standard PyTorch serialization (`state_dict`).
SWAG and Laplace include their computed statistics in the state dict.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    # Save MC Dropout (just save the base model)
    mc_path = os.path.join(tmpdir, "mc_dropout.pt")
    torch.save(mc_base_model.state_dict(), mc_path)
    print(f"MC Dropout saved ({os.path.getsize(mc_path):,} bytes)")
    
    # Save Deep Ensemble (save all models)
    ens_path = os.path.join(tmpdir, "ensemble.pt")
    torch.save(ensemble.state_dict(), ens_path)
    print(f"Deep Ensemble saved ({os.path.getsize(ens_path):,} bytes)")
    
    # Save SWAG (includes collected statistics)
    swag_path = os.path.join(tmpdir, "swag.pt")
    torch.save(swag.state_dict(), swag_path)
    print(f"SWAG saved ({os.path.getsize(swag_path):,} bytes)")
    
    # Save Laplace (includes posterior precision)
    laplace_path = os.path.join(tmpdir, "laplace.pt")
    torch.save(laplace.state_dict(), laplace_path)
    print(f"Laplace saved ({os.path.getsize(laplace_path):,} bytes)")
    
    # Load and verify SWAG
    loaded_swag_model = DropoutCNN(dropout_rate=0.0)
    loaded_swag = SWAG(loaded_swag_model, num_samples=20)
    loaded_swag.load_state_dict(torch.load(swag_path, weights_only=False))
    
    print(f"\nLoaded SWAG: n_models = {loaded_swag.n_models}")

In [ ]:
# Production inference example
def predict_with_uncertainty(model, inputs, threshold=0.5):
    """
    Production Bayesian prediction with uncertainty.
    
    Args:
        model: Bayesian model (MCDropout, DeepEnsemble, SWAG, etc.)
        inputs: Input tensor
        threshold: Uncertainty threshold for abstention
    
    Returns:
        dict with predictions, confidence, uncertainty, and abstain flags
    """
    mean, variance = model.predict(inputs)
    
    probs = mean
    predictions = probs.argmax(dim=1)
    confidence = probs.max(dim=1).values
    uncertainty = variance.mean(dim=1)
    
    # Abstain if uncertainty is too high
    abstain = uncertainty > threshold
    
    return {
        'predictions': predictions,
        'confidence': confidence,
        'uncertainty': uncertainty,
        'abstain': abstain,
        'probabilities': probs
    }

# Example
test_batch, test_batch_labels = next(iter(test_loader))
test_batch = test_batch[:5].to(device)

result = predict_with_uncertainty(ensemble, test_batch, threshold=0.02)

print("Production Inference Example (Deep Ensemble):")
print("=" * 70)
for i in range(5):
    pred = result['predictions'][i].item()
    conf = result['confidence'][i].item()
    unc = result['uncertainty'][i].item()
    abstain = result['abstain'][i].item()
    true = test_batch_labels[i].item()
    status = "ABSTAIN" if abstain else ("correct" if pred == true else "WRONG")
    
    print(f"  {CLASS_NAMES[pred]:10s} (conf={conf:.2f}, unc={unc:.4f}) | true: {CLASS_NAMES[true]:10s} [{status}]")

## Summary

### Method Selection Guide

| Scenario | Recommended Method |
|----------|--------------------|
| Fast inference, existing model | MC Dropout |
| Best uncertainty quality | Deep Ensembles |
| Single training run | SWAG |
| Post-hoc uncertainty | Laplace Approximation |
| Research / principled Bayesian | Variational BNN |

### Key Takeaways

1. **MC Dropout** is the simplest - just enable dropout at test time
2. **Deep Ensembles** give the best uncertainty but cost 5-10x training
3. **SWAG** offers a good balance - collect weight statistics during training
4. **Laplace** works with any pre-trained model (post-hoc)
5. **Variational BNNs** learn weight distributions but need model redesign

### Uncertainty Quality

Good uncertainty estimates should:
- Be higher for incorrect predictions (high AUROC)
- Be higher for OOD samples
- Decompose into epistemic (reducible) and aleatoric (irreducible) components

### Next Steps

- Combine with calibration for better confidence estimates
- Use for selective prediction (abstain on high uncertainty)
- Apply to active learning (query high-uncertainty samples)
- See `04_active_learning.ipynb` for active learning with uncertainty